# 📁 Notebook 07 — RSNA Data Preparation
## FedMedSeg Phase 2: Semantic Segmentation

---

This notebook prepares the dataset for Phase 2 training.

**We will:**
1. Download / verify the RSNA Pneumonia Detection dataset
2. Extract a balanced 5,000-image subset
3. Visualize the bounding box → binary mask conversion
4. Confirm the data pipeline is working correctly

---
**Dataset source:** [RSNA Pneumonia Detection Challenge (Kaggle)](https://www.kaggle.com/c/rsna-pneumonia-detection-challenge/data)

## ⬇️ Step 1 — Download the Dataset

### Option A: By Kaggle API (Recommended)
Run the cells below after placing your `kaggle.json` API key at `~/.kaggle/kaggle.json`.

### Option B: Manual Download
1. Go to: https://www.kaggle.com/c/rsna-pneumonia-detection-challenge/data
2. Download `stage_2_train_images.zip` and `stage_2_train_labels.csv`
3. Extract to: `data/rsna_pneumonia/`

In [1]:
import subprocess, os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RSNA_DIR     = PROJECT_ROOT / 'data' / 'rsna_pneumonia'
RSNA_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project Root: {PROJECT_ROOT}')
print(f'RSNA Data Dir: {RSNA_DIR}')

# Option A: Kaggle API download
# Uncomment the lines below if you have the Kaggle API set up

subprocess.run([
    'kaggle', 'datasets', 'download',
    '-d', 'blank3030/rsnapneumoniadetectionchallenge',
    '-p', str(RSNA_DIR)
], check=True)
subprocess.run(['unzip', '-q', str(RSNA_DIR / 'rsnapneumoniadetectionchallenge.zip'),
                '-d', str(RSNA_DIR)], check=True)

print('\n✓ Skipping download — assumes data is already in RSNA_DIR')

Project Root: /home/rajat/Documents/Project/FedMedSeg
RSNA Data Dir: /home/rajat/Documents/Project/FedMedSeg/data/rsna_pneumonia
Dataset URL: https://www.kaggle.com/datasets/blank3030/rsnapneumoniadetectionchallenge
License(s): unknown
Resuming from 356515840 bytes (3575112111 bytes left)...


100%|██████████| 3.66G/3.66G [19:59<00:00, 2.98MB/s]




✓ Skipping download — assumes data is already in RSNA_DIR


## 🔍 Step 2 — Verify Dataset Files

In [ ]:
import pandas as pd

labels_csv  = RSNA_DIR / 'stage_2_train_labels.csv'
images_dir  = RSNA_DIR / 'stage_2_train_images'

assert labels_csv.exists(),  f'Missing: {labels_csv}'
assert images_dir.exists(),  f'Missing: {images_dir}'

df = pd.read_csv(labels_csv)
print('=== stage_2_train_labels.csv ===')
print(f'Total rows:     {len(df)}')
print(f'Unique patients: {df["patientId"].nunique()}')
print(f'Column names:   {list(df.columns)}')
print(f'\nClass distribution:')
print(df['Target'].value_counts().rename({0: 'Normal', 1: 'Pneumonia'}))
print(f'\nFirst 5 rows:')
df.head()

In [ ]:
# Count DICOM files
dcm_files = list(images_dir.glob('*.dcm'))
print(f'DICOM files found: {len(dcm_files)}')
print(f'Example: {dcm_files[0].name if dcm_files else "None found"}')

## ✂️ Step 3 — Extract 5,000-Image Balanced Subset

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from segmentation.prepare_subset import prepare_balanced_subset

train_df, val_df = prepare_balanced_subset(
    labels_csv      = labels_csv,
    output_dir      = RSNA_DIR / 'subset',
    pneumonia_count = 2500,
    normal_count    = 2500,
    train_ratio     = 0.80,
    seed            = 42,
    images_dir      = images_dir,
)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, split_df, title in zip(axes, [train_df, val_df], ['Training Set', 'Validation Set']):
    counts = split_df['label'].map({0: 'Normal', 1: 'Pneumonia'}).value_counts()
    ax.bar(counts.index, counts.values, color=['#4CAF50', '#F44336'])
    ax.set_title(f'{title} ({len(split_df)} images)', fontsize=13, fontweight='bold')
    ax.set_ylabel('Count')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 10, str(v), ha='center', fontweight='bold')

plt.suptitle('Class Distribution in Subset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'results' / 'segmentation' / 'subset_distribution.png', dpi=120)
plt.show()
print('✓ Class distribution saved')

## 🎭 Step 4 — Visualize Bounding Box → Mask Conversion

This is the most important verification step. We need to check that the white rectangles in our generated masks correctly overlap the pneumonia region in the X-ray.

In [ ]:
import numpy as np
from segmentation.dataset_rsna import load_dicom_as_pil, generate_mask

# Get a few pneumonia patient IDs
pneumonia_patients = train_df[train_df['label'] == 1]['patientId'].tolist()[:6]
all_labels         = pd.read_csv(labels_csv)

fig, axes = plt.subplots(6, 3, figsize=(15, 30))
fig.suptitle('X-ray | Ground Truth Mask | Overlay\n(White region = Pneumonia bounding box)',
             fontsize=14, fontweight='bold', y=1.01)

for row_idx, pid in enumerate(pneumonia_patients):
    dcm_path  = images_dir / f'{pid}.dcm'
    bbox_rows = all_labels[all_labels['patientId'] == pid]

    # Load image and generate mask
    image = load_dicom_as_pil(dcm_path)
    image_resized = image.resize((224, 224))
    mask  = generate_mask(bbox_rows, target_size=224)

    img_arr  = np.array(image_resized) / 255.0
    mask_arr = np.array(mask)  / 255.0

    # Overlay (red mask on x-ray)
    overlay = img_arr.copy()
    overlay[:, :, 0] = np.where(mask_arr > 0.5, 1.0, overlay[:, :, 0])
    overlay[:, :, 1] = np.where(mask_arr > 0.5, 0.0, overlay[:, :, 1])
    overlay[:, :, 2] = np.where(mask_arr > 0.5, 0.0, overlay[:, :, 2])

    axes[row_idx, 0].imshow(img_arr); axes[row_idx, 0].set_title(f'{pid[:8]}...')
    axes[row_idx, 1].imshow(mask_arr, cmap='gray'); axes[row_idx, 1].set_title('Mask')
    axes[row_idx, 2].imshow(overlay); axes[row_idx, 2].set_title('Overlay')

    for ax in axes[row_idx]: ax.axis('off')

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'results' / 'segmentation' / 'mask_verification.png', dpi=120, bbox_inches='tight')
plt.show()
print('✓ Mask verification saved!')

## ✅ Step 5 — Test the Full DataLoader Pipeline

In [ ]:
import torchvision.transforms as T
from torch.utils.data import DataLoader
from segmentation.dataset_rsna import RSNAPneumoniaDataset

transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

dataset = RSNAPneumoniaDataset(
    rsna_root     = str(RSNA_DIR),
    subset_csv    = str(RSNA_DIR / 'subset' / 'train_subset.csv'),
    img_transform = transform,
    augment       = True,
)

loader = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=2)

# Load one batch
images, masks = next(iter(loader))

print('=== DataLoader Test ===')
print(f'Image batch shape: {images.shape}   (B, C, H, W)')
print(f'Mask  batch shape: {masks.shape}    (B, 1, H, W)')
print(f'Image value range: [{images.min():.2f}, {images.max():.2f}]  (after ImageNet normalization)')
print(f'Mask  value range: [{masks.min():.2f}, {masks.max():.2f}]   (binary 0 or 1)')
print('\n✓ DataLoader is working correctly!')

## 🎉 Data Preparation Complete!

You now have:
- `data/rsna_pneumonia/subset/train_subset.csv` — 4,000 training patients
- `data/rsna_pneumonia/subset/val_subset.csv` — 1,000 validation patients
- `data/rsna_pneumonia/subset/subset_summary.json` — Reproducibility info

**Next:** Open `08-Segmentation-Training.ipynb` to begin training!